# Error generator polynomials
In this tutorial we provide an introduction to the functionality available through pyGSTi's `errgenpolytools` module.

The error generator propagation framework described in the companion tutorial {doc}`ErrorGeneratorPropagation` makes it possible to propagate sparse Markovian error generators through Clifford circuits and to construct efficient approximations to the resulting noisy dynamics. The `errgenpolytools` module builds directly on top of that framework and allows one to go one step further: instead of working only with *numerical* propagated error generator rates, we can construct **symbolic polynomial representations** of quantities of interest such as

- effective end-of-circuit error generators,
- corrections to computational basis measurement probabilities, and
- corrections to Pauli observable expectation values.

These polynomial representations are useful when one wants to:

- evaluate the same circuit quantities many times for different noise-parameter values,
- connect circuit-level propagated quantities back to model-level parameters,
- efficiently construct multiple observable corrections while reusing shared intermediate results.

The `errgenpolytools` module provides this additional symbolic layer by representing relevant rates and observable corrections as instances of pyGSTi's `Polynomial` class.

Please note: the functionality described here requires the error generator propagation framework, and thus also requires the `stim` python package.

In [1]:
import pygsti
import stim
import numpy as np
from itertools import product

from pygsti.tools import errgenproptools as eprop
from pygsti.tools import errgenpolytools as epoly
from pygsti.tools.lindbladtools import random_CPTP_error_generator_rates
from pygsti.errorgenpropagation.errorpropagator import ErrorGeneratorPropagator

## Setup
As in the error generator propagation tutorial, we begin by constructing a target model, a noisy model, and an example random Clifford circuit.

In [2]:
num_qubits = 4
gate_names = ['Gcphase', 'Gxpi2', 'Gypi2']
availability = {'Gcphase': [(0, 1), (1, 2), (2, 3), (3, 0)]}
pspec = pygsti.processors.QubitProcessorSpec(num_qubits, gate_names, availability=availability)
target_model = pygsti.models.create_crosstalk_free_model(processor_spec=pspec)

For the noisy model we will use a crosstalk-free model with local H+S error generators. To simplify the later discussion of how to handle the aggregation of error generator rates associated with shared model parameters, we'll construct a model where all instances of a given gate type share the same error generator parameters. This is not a fundamental restriction, however; the functionality described in this tutorial is compatible with more general model parameter structures.

In [3]:
error_rates_dict_shared = {
    'Gcphase': random_CPTP_error_generator_rates(2, errorgen_types=('H', 'S'), label_type='local', seed=1234),
    'Gxpi2': random_CPTP_error_generator_rates(1, errorgen_types=('H', 'S'), label_type='local', seed=1235),
    'Gypi2': random_CPTP_error_generator_rates(1, errorgen_types=('H', 'S'), label_type='local', seed=1236)
}
error_model = pygsti.models.create_crosstalk_free_model(pspec, lindblad_error_coeffs=error_rates_dict_shared)

We also create an `ErrorGeneratorPropagator`, which is the object responsible for producing the propagation data structures used by `errgenpolytools`.

In [4]:
errorgen_propagator = ErrorGeneratorPropagator(error_model)

Finally, let us generate a random example circuit.

In [5]:
c = pygsti.algorithms.randomcircuit.create_random_circuit(
    pspec, 4, sampler='edgegrab', samplerargs=[0.4], rand_state=12345
)
print(c)

Qubit 0 ---|Gypi2|-|Gxpi2|-|Gxpi2|-|Gypi2|---
Qubit 1 ---|Gypi2|-|Gypi2|-|Gypi2|-|Gxpi2|---
Qubit 2 ---|Gypi2|-|Gypi2|-|Gypi2|-| C3  |---
Qubit 3 ---|Gypi2|-|Gypi2|-|Gxpi2|-| C2  |---



For later use we will also create the corresponding `stim.Tableau`.

In [6]:
tableau = c.convert_to_stim_tableau()

## From Propagation Maps to Polynomial Variables
The first step in constructing symbolic polynomials is to create a map from elementary error generators to corresponding polynomial variables. The relevant starting point are the error generator transform maps produced by the `ErrorGeneratorPropagator`.

In [7]:
errorgen_transform_map = errorgen_propagator.errorgen_transform_map(c)
errorgen_transform_maps = errorgen_propagator.errorgen_transform_maps(c)

These maps give the input-output relationship for an elementary error generator rate following propagation to the end of the circuit `c`. These methods return a dictionary (or dictionaries) with the following structure: Keys are tuples of the form (<original_errorgen_label>, <layer_index>), and values are of the form (<final_errorgen_label>, <overall_phase>), where overall_phase corresponds to the overall sign accumulated on the final error generator rate as a result of propagation. The former method gives this as a single aggregated dictionary for all circuit layers, while the latter gives a list of dictionaries, one per circuit layer.

The method `error_generator_to_polynomial_variable_maps` constructs a map from `(<input_error_generator>, <layer_index>)` pairs to integer variable indices for use in polynomials.

In [8]:
errorgen_to_var_map, var_to_errorgen_map = epoly.error_generator_to_polynomial_variable_maps(
    errorgen_transform_map, return_reverse=True
)

The forward map tells us which variable index corresponds to each propagated input error generator, while the reverse map lets us interpret a polynomial variable index in terms of the corresponding error generator and circuit layer.

In [9]:
list(errorgen_to_var_map.items())[:5]

[(((H, (stim.PauliString("+X___"),)), 1), 0),
 (((H, (stim.PauliString("+Y___"),)), 1), 1),
 (((H, (stim.PauliString("+Z___"),)), 1), 2),
 (((S, (stim.PauliString("+X___"),)), 1), 3),
 (((S, (stim.PauliString("+Y___"),)), 1), 4)]

In [10]:
list(var_to_errorgen_map.items())[:5]

[(0, ((H, (stim.PauliString("+X___"),)), 1)),
 (1, ((H, (stim.PauliString("+Y___"),)), 1)),
 (2, ((H, (stim.PauliString("+Z___"),)), 1)),
 (3, ((S, (stim.PauliString("+X___"),)), 1)),
 (4, ((S, (stim.PauliString("+Y___"),)), 1))]

To actually *evaluate* polynomials later on, we will also need a numerical vector of values for these variables. The helper function `construct_polynomial_parameter_vector_from_propagator` constructs exactly this vector.

In [11]:
poly_paramvec = epoly.construct_polynomial_parameter_vector_from_propagator(
    errorgen_propagator, var_to_errorgen_map, c
)
poly_paramvec[:10]

array([-0.01358528, -0.0115617 ,  0.00323786,  0.00088949,  0.00074695,
        0.00047002, -0.01358528, -0.0115617 ,  0.00323786,  0.00088949])

This vector can be fed directly into the `.evaluate(...)` method of a `Polynomial`.

## Constructing Polynomial Magnus Expansions
The first major construction provided by `errgenpolytools` is a symbolic Magnus approximation to the effective end-of-circuit error generator. The function `magnus_symbolic_polynomial` takes as input the layer-by-layer transform maps and a variable map and returns a dictionary whose keys are final `LocalStimErrorgenLabel`s and whose values are `Polynomial` objects giving the corresponding rates.

### First-order Magnus
We begin with the first-order Magnus approximation.

In [12]:
first_order_magnus_polys = epoly.magnus_symbolic_polynomial(
    errorgen_transform_maps, errorgen_to_var_map, magnus_order=1
)

Let us inspect a few representative terms.

In [13]:
for i, (lbl, poly) in enumerate(first_order_magnus_polys.items()):
    print(lbl, "->", poly)
    if i == 4:
        break

H(+Z___) -> -1.000x0 + -1.000x24 + -1.000x48 + 1.000x74
H(+Y___) -> -1.000x1 + -1.000x26 + 1.000x49 + 1.000x73
H(+X___) -> -1.000x2 + 1.000x25 + 1.000x50 + 1.000x72
S(+Z___) -> 1.000x3 + 1.000x27 + 1.000x51 + 1.000x77
S(+Y___) -> 1.000x4 + 1.000x29 + 1.000x52 + 1.000x76


Because these are ordinary `Polynomial` objects, we can evaluate them on the parameter vector constructed above.

In [14]:
first_five = list(first_order_magnus_polys.items())[:5]
for lbl, poly in first_five:
    print(lbl, "->", poly.evaluate(poly_paramvec))

H(+Z___) -> (0.03501645162848567+0j)
H(+Y___) -> (0.012876308836241187+0j)
H(+X___) -> (-0.02439407675380065+0j)
S(+Z___) -> (0.0019934452674093546+0j)
S(+Y___) -> (0.0026453017369992035+0j)


We can compare these values against the first-order BCH/Magnus propagation result computed using numerical rates.

In [15]:
propagated_errorgen_layer_first_order = errorgen_propagator.propagate_errorgens_bch(c, bch_order=1)
for lbl, poly in first_five:
    print(lbl)
    print("  polynomial evaluation:", poly.evaluate(poly_paramvec))
    print("  numerical BCH result:", propagated_errorgen_layer_first_order[lbl])

H(+Z___)
  polynomial evaluation: (0.03501645162848567+0j)
  numerical BCH result: 0.03501645162848566
H(+Y___)
  polynomial evaluation: (0.012876308836241187+0j)
  numerical BCH result: 0.012876308836241187
H(+X___)
  polynomial evaluation: (-0.02439407675380065+0j)
  numerical BCH result: -0.024394076753800654
S(+Z___)
  polynomial evaluation: (0.0019934452674093546+0j)
  numerical BCH result: 0.0019934452674093546
S(+Y___)
  polynomial evaluation: (0.0026453017369992035+0j)
  numerical BCH result: 0.0026453017369992035


### Second-order Magnus
The symbolic Magnus construction also supports second order.

In [16]:
second_order_magnus_polys = epoly.magnus_symbolic_polynomial(
    errorgen_transform_maps, errorgen_to_var_map, magnus_order=2
)

At second order one may observe the appearance of new contributions to the effective elementary error generator terms arising from non-commutativity of the original elementary error generators.

In [17]:
for i, (lbl, poly) in enumerate(second_order_magnus_polys.items()):
    print(lbl, "->", poly)
    if i == 4:
        break

H(+Z___) -> -1.000x0 + -1.000x1x25 + -1.000x1x50 + -1.000x1x72 + -1.000x2x26 + 1.000x2x49 + 1.000x2x73 + -1.000x24 + -1.000x25x49 + -1.000x25x73 + -1.000x26x50 + -1.000x26x72 + -1.000x48 + 1.000x49x72 + -1.000x50x73 + 1.000x74
H(+Y___) -> 1.000x0x25 + 1.000x0x50 + 1.000x0x72 + -1.000x1 + 1.000x2x24 + 1.000x2x48 + -1.000x2x74 + 1.000x24x50 + 1.000x24x72 + -1.000x25x48 + 1.000x25x74 + -1.000x26 + 1.000x48x72 + 1.000x49 + 1.000x50x74 + 1.000x73
H(+X___) -> 1.000x0x26 + -1.000x0x49 + -1.000x0x73 + -1.000x1x24 + -1.000x1x48 + 1.000x1x74 + -1.000x2 + -1.000x24x49 + -1.000x24x73 + 1.000x25 + -1.000x26x48 + 1.000x26x74 + -1.000x48x73 + -1.000x49x74 + 1.000x50 + 1.000x72
S(+Z___) -> 1.000x3 + 1.000x27 + 1.000x51 + 1.000x77
S(+Y___) -> 1.000x4 + 1.000x29 + 1.000x52 + 1.000x76


And again we can compare evaluated symbolic results against the numerical second-order BCH result.

In [18]:
propagated_errorgen_layer_second_order = errorgen_propagator.propagate_errorgens_bch(c, bch_order=2)
first_five_second_order = list(second_order_magnus_polys.items())[:5]
for lbl, poly in first_five_second_order:
    print(lbl)
    print("  polynomial evaluation:", poly.evaluate(poly_paramvec))
    print("  numerical BCH result:", propagated_errorgen_layer_second_order[lbl])

H(+Z___)
  polynomial evaluation: (0.034402086205174986+0j)
  numerical BCH result: 0.03440208620517498
H(+Y___)
  polynomial evaluation: (0.013434109555842857+0j)
  numerical BCH result: 0.013434109555842856
H(+X___)
  polynomial evaluation: (-0.024944905435624717+0j)
  numerical BCH result: -0.024944905435624727
S(+Z___)
  polynomial evaluation: (0.0019934452674093546+0j)
  numerical BCH result: 0.0019934452674093546
S(+Y___)
  polynomial evaluation: (0.0026453017369992035+0j)
  numerical BCH result: 0.0026453017369992035


Please note that the symbolic Magnus implementation currently supports first and second order. 

## Constructing Taylor-Series Polynomials
Once we have a symbolic representation of the effective end-of-circuit error generator, we can form the Taylor series approximation to its exponential using `error_generator_taylor_expansion_symbolic_polynomial`.

This function returns a list of dictionaries, one per Taylor order (excluding zeroth order), with the same key structure as the Magnus dictionary.

### First-order Taylor expansion

In [19]:
first_order_taylor_terms = epoly.error_generator_taylor_expansion_symbolic_polynomial(
    first_order_magnus_polys, errorgen_to_var_map, order=1
)
first_order_taylor_polys = first_order_taylor_terms[0]

In [20]:
for i, (lbl, poly) in enumerate(first_order_taylor_polys.items()):
    print(lbl, "->", poly)
    if i == 4:
        break

H(+Z___) -> -1.000x0 + -1.000x24 + -1.000x48 + 1.000x74
H(+Y___) -> -1.000x1 + -1.000x26 + 1.000x49 + 1.000x73
H(+X___) -> -1.000x2 + 1.000x25 + 1.000x50 + 1.000x72
S(+Z___) -> 1.000x3 + 1.000x27 + 1.000x51 + 1.000x77
S(+Y___) -> 1.000x4 + 1.000x29 + 1.000x52 + 1.000x76


### Second-order Taylor expansion

In [21]:
second_order_taylor_terms = epoly.error_generator_taylor_expansion_symbolic_polynomial(
    first_order_magnus_polys, errorgen_to_var_map, order=2
)
second_order_taylor_polys = second_order_taylor_terms[1]

In [22]:
for i, (lbl, poly) in enumerate(second_order_taylor_polys.items()):
    print(lbl, "->", poly)
    if i == 4:
        break

S(+Z___) -> 1.000x0^2 + 2.000x0x24 + 2.000x0x48 + -2.000x0x74 + -1.000x3^2 + -1.000x3x4 + -1.000x3x5 + -1.000x3x9 + -1.000x3x10 + -1.000x3x11 + -1.000x3x15 + -1.000x3x16 + -1.000x3x17 + -1.000x3x21 + -1.000x3x22 + -1.000x3x23 + -2.000x3x27 + -1.000x3x28 + -1.000x3x29 + -1.000x3x33 + -1.000x3x34 + -1.000x3x35 + -1.000x3x39 + -1.000x3x40 + -1.000x3x41 + -1.000x3x45 + -1.000x3x46 + -1.000x3x47 + -2.000x3x51 + -1.000x3x52 + -1.000x3x53 + -1.000x3x57 + -1.000x3x58 + -1.000x3x59 + -1.000x3x63 + -1.000x3x64 + -1.000x3x65 + -1.000x3x69 + -1.000x3x70 + -1.000x3x71 + -1.000x3x75 + -1.000x3x76 + -2.000x3x77 + -1.000x3x81 + -1.000x3x82 + -1.000x3x83 + -1.000x3x99 + -1.000x3x100 + -1.000x3x101 + -1.000x3x102 + -1.000x3x103 + -1.000x3x104 + -1.000x3x105 + -1.000x3x106 + -1.000x3x107 + -1.000x3x108 + -1.000x3x109 + -1.000x3x110 + -1.000x3x111 + -1.000x3x112 + -1.000x3x113 + 1.000x4x5 + -1.000x4x27 + 1.000x4x28 + -1.000x4x51 + 1.000x4x53 + 1.000x4x75 + -1.000x4x77 + -1.000x5x27 + 1.000x5x29 + -1.000x5

We can compare evaluated symbolic Taylor terms against the corresponding Taylor expansion with numerical rates from `errgenproptools`.

In [23]:
numeric_taylor_order_1 = eprop.error_generator_taylor_expansion(propagated_errorgen_layer_first_order, order=1)[0]
numeric_taylor_order_2 = eprop.error_generator_taylor_expansion(
    propagated_errorgen_layer_first_order, order=2, truncation_threshold=-1
)[1]

In [24]:
for lbl, poly in list(first_order_taylor_polys.items())[:5]:
    print(lbl)
    print("  polynomial evaluation:", poly.evaluate(poly_paramvec))
    print("  numerical Taylor term:", numeric_taylor_order_1[lbl])

H(+Z___)
  polynomial evaluation: (0.03501645162848567+0j)
  numerical Taylor term: 0.03501645162848566
H(+Y___)
  polynomial evaluation: (0.012876308836241187+0j)
  numerical Taylor term: 0.012876308836241187
H(+X___)
  polynomial evaluation: (-0.02439407675380065+0j)
  numerical Taylor term: -0.024394076753800654
S(+Z___)
  polynomial evaluation: (0.0019934452674093546+0j)
  numerical Taylor term: 0.0019934452674093546
S(+Y___)
  polynomial evaluation: (0.0026453017369992035+0j)
  numerical Taylor term: 0.0026453017369992035


In [25]:
for lbl, poly in list(second_order_taylor_polys.items())[:5]:
    print(lbl)
    print("  polynomial evaluation:", poly.evaluate(poly_paramvec))
    print("  numerical Taylor term:", numeric_taylor_order_2[lbl])

S(+Z___)
  polynomial evaluation: (0.0010878105719031353+0j)
  numerical Taylor term: 0.0010878105719031373
H(+X___)
  polynomial evaluation: (0.00183543404254742+0j)
  numerical Taylor term: (0.0018354340425474221+0j)
C(+Y___,+Z___)
  polynomial evaluation: (0.0004508826455176821+0j)
  numerical Taylor term: (0.00045088264551768206+0j)
H(+Y___)
  polynomial evaluation: (-0.0009705565428003129+0j)
  numerical Taylor term: (-0.0009705565428003125+0j)
C(+X___,+Z___)
  polynomial evaluation: (-0.0008541940086710272+0j)
  numerical Taylor term: (-0.0008541940086710271+0j)


## Probability Correction Polynomials
A major use case for `errgenpolytools` is the construction of symbolic corrections to the
probabilities of computational basis measurement outcomes.

The function `stabilizer_probability_correction_symbolic_polynomial` returns a `Polynomial`
corresponding to the correction to a specified bitstring probability.

### Single-bitstring probability correction
Let us compute the correction polynomial for the output bitstring `'0000'`.

In [26]:
prob_corr_poly_order_1 = epoly.stabilizer_probability_correction_symbolic_polynomial(
    first_order_magnus_polys, errorgen_to_var_map, tableau, '0000', order=1
)
print(prob_corr_poly_order_1)

0.125x4 + 0.125x5 + 0.125x28 + 0.125x29 + 0.125x52 + 0.125x53 + 0.125x75 + 0.125x76


Evaluating this polynomial gives the numerical first-order correction.

In [27]:
prob_corr_poly_order_1.evaluate(poly_paramvec)

(0.0006445256662318686+0j)

We can compare this against the numerical correction computed using `errgenproptools`.

In [28]:
prob_corr_numeric_order_1 = eprop.stabilizer_probability_correction(
    propagated_errorgen_layer_first_order, tableau, '0000', order=1
)
print(prob_corr_numeric_order_1)

0.0006445256662318686


Similarly for second order:

In [29]:
prob_corr_poly_order_2 = epoly.stabilizer_probability_correction_symbolic_polynomial(
    first_order_magnus_polys, errorgen_to_var_map, tableau, '0000', order=2
)
prob_corr_numeric_order_2 = eprop.stabilizer_probability_correction(
    propagated_errorgen_layer_first_order, tableau, '0000', order=2
)
print(prob_corr_poly_order_2.evaluate(poly_paramvec))
print(prob_corr_numeric_order_2)

(0.0006329645074924+0j)
0.0006329645074924001


### Bulk probability corrections
When one wants probability corrections for many bitstrings, the bulk interface can reuse
intermediate results and is typically much more efficient.

In [30]:
bitstrings_4Q = [''.join(bs) for bs in product(['0', '1'], repeat=4)]
bulk_prob_corr_polys = epoly.bulk_stabilizer_probability_correction_symbolic_polynomial(
    first_order_magnus_polys, errorgen_to_var_map, tableau, bitstrings_4Q, order=1
)

In [31]:
for bs, poly in zip(bitstrings_4Q, bulk_prob_corr_polys):
    print(bs, "->", poly)
    print('------')

0000 -> 0.125x4 + 0.125x5 + 0.125x28 + 0.125x29 + 0.125x52 + 0.125x53 + 0.125x75 + 0.125x76
------
0001 -> 0.125x4 + 0.125x5 + 0.125x28 + 0.125x29 + 0.125x52 + 0.125x53 + 0.125x75 + 0.125x76
------
0010 -> 0.125x4 + 0.125x5 + 0.125x28 + 0.125x29 + 0.125x52 + 0.125x53 + 0.125x75 + 0.125x76
------
0011 -> 0.125x4 + 0.125x5 + 0.125x28 + 0.125x29 + 0.125x52 + 0.125x53 + 0.125x75 + 0.125x76
------
0100 -> 0.125x4 + 0.125x5 + 0.125x28 + 0.125x29 + 0.125x52 + 0.125x53 + 0.125x75 + 0.125x76
------
0101 -> 0.125x4 + 0.125x5 + 0.125x28 + 0.125x29 + 0.125x52 + 0.125x53 + 0.125x75 + 0.125x76
------
0110 -> 0.125x4 + 0.125x5 + 0.125x28 + 0.125x29 + 0.125x52 + 0.125x53 + 0.125x75 + 0.125x76
------
0111 -> 0.125x4 + 0.125x5 + 0.125x28 + 0.125x29 + 0.125x52 + 0.125x53 + 0.125x75 + 0.125x76
------
1000 -> -0.125x4 + -0.125x5 + 0.250x8 + 0.250x13 + 0.250x20 + -0.125x28 + -0.125x29 + 0.250x30 + 0.250x37 + 0.250x42 + -0.125x52 + -0.125x53 + -0.250x56 + 0.250x61 + 0.250x66 + -0.125x75 + -0.125x76 + 0.250x7

### Recovering approximate probabilities
To obtain an approximate noisy probability, we add the correction to the ideal stabilizer
probability.

In [32]:
ideal_prob_0000 = eprop.stabilizer_probability(tableau, '0000')
approx_prob_0000 = ideal_prob_0000 + prob_corr_poly_order_2.evaluate(poly_paramvec)
print("Ideal probability:", ideal_prob_0000)
print("Approximate noisy probability:", approx_prob_0000)

Ideal probability: 0
Approximate noisy probability: (0.0006329645074924+0j)


For a few-qubit example we can compare this against the exact dense forward simulation.

In [33]:
exact_prob_0000 = error_model.sim.probs(c)['0000']
print("Exact noisy probability:", exact_prob_0000)
print("Absolute error:", abs(exact_prob_0000 - approx_prob_0000))

Exact noisy probability: 0.0006326881398648589
Absolute error: 2.763676275410482e-07


## Pauli Expectation Correction Polynomials
In addition to measurement probabilities, `errgenpolytools` can also construct symbolic correction
polynomials for expectation values of Pauli observables.

The relevant functions are:
- `stabilizer_pauli_expectation_correction_symbolic_polynomial`
- `bulk_stabilizer_pauli_expectation_correction_symbolic_polynomial`

### Single-Pauli expectation correction
Let us compute the correction polynomial for the observable `IIXZ`.

In [34]:
pauli = stim.PauliString('IIXZ')
pauli_corr_poly_order_1 = epoly.stabilizer_pauli_expectation_correction_symbolic_polynomial(
    first_order_magnus_polys, errorgen_to_var_map, tableau, pauli, order=1
)
print(pauli_corr_poly_order_1)

2.000x16 + 2.000x17 + 2.000x39 + 2.000x40 + 2.000x64 + 2.000x65 + 2.000x99 + 2.000x100 + 2.000x103 + 2.000x104 + 2.000x106 + 2.000x109 + 2.000x110 + 2.000x113


In [35]:
pauli_corr_poly_order_1.evaluate(poly_paramvec)

(0.0590231105638102+0j)

Compare this to the numerical correction.

In [36]:
pauli_corr_numeric_order_1 = eprop.stabilizer_pauli_expectation_correction(
    propagated_errorgen_layer_first_order, tableau, pauli, order=1
)
print(pauli_corr_numeric_order_1)

0.05902311056381019


And again at second order:

In [37]:
pauli_corr_poly_order_2 = epoly.stabilizer_pauli_expectation_correction_symbolic_polynomial(
    first_order_magnus_polys, errorgen_to_var_map, tableau, pauli, order=2
)
pauli_corr_numeric_order_2 = eprop.stabilizer_pauli_expectation_correction(
    propagated_errorgen_layer_first_order, tableau, pauli, order=2
)
print(pauli_corr_poly_order_2.evaluate(poly_paramvec))
print(pauli_corr_numeric_order_2)

(0.06136301287783426+0j)
0.06136301287783421


## Custom Variable Labels for Polynomial Output
By default, the string representation of a `Polynomial` uses generic variable names such as
`x0`, `x1`, `x2`, and so on. When working with error generator polynomials it is often useful to
replace these generic names with labels that make clear which propagated error generator parameter
each variable corresponds to.

Because the variables used in `errgenpolytools` are indexed by the maps returned from
`error_generator_to_polynomial_variable_maps` (or its aggregated variants), we can construct a
dictionary mapping variable indices to more descriptive labels and pass this into the
`Polynomial.to_string(...)` method.

For example, let us revisit one of the symbolic expectation-value correction polynomials from above.

In [38]:
custom_label_poly = epoly.stabilizer_pauli_expectation_correction_symbolic_polynomial(
    first_order_magnus_polys,
    errorgen_to_var_map,
    tableau,
    stim.PauliString('IIXZ'),
    order=2
)
print(custom_label_poly)

2.000x13^2 + 4.000x13x37 + 4.000x13x61 + 4.000x13x88 + 4.000x13x94 + 2.000x14^2 + 4.000x14x36 + -4.000x14x62 + -4.000x14x85 + -4.000x14x95 + 2.000x16 + -2.000x16^2 + -4.000x16x17 + -4.000x16x39 + -4.000x16x40 + -4.000x16x64 + -4.000x16x65 + -4.000x16x99 + -4.000x16x100 + -4.000x16x103 + -4.000x16x104 + -4.000x16x106 + -4.000x16x109 + -4.000x16x110 + -4.000x16x113 + 2.000x17 + -2.000x17^2 + -4.000x17x39 + -4.000x17x40 + -4.000x17x64 + -4.000x17x65 + -4.000x17x99 + -4.000x17x100 + -4.000x17x103 + -4.000x17x104 + -4.000x17x106 + -4.000x17x109 + -4.000x17x110 + -4.000x17x113 + 2.000x36^2 + -4.000x36x62 + -4.000x36x85 + -4.000x36x95 + 2.000x37^2 + 4.000x37x61 + 4.000x37x88 + 4.000x37x94 + 2.000x39 + -2.000x39^2 + -4.000x39x40 + -4.000x39x64 + -4.000x39x65 + -4.000x39x99 + -4.000x39x100 + -4.000x39x103 + -4.000x39x104 + -4.000x39x106 + -4.000x39x109 + -4.000x39x110 + -4.000x39x113 + 2.000x40 + -2.000x40^2 + -4.000x40x64 + -4.000x40x65 + -4.000x40x99 + -4.000x40x100 + -4.000x40x103 + -4.000x4

The default output is compact, but it does not tell us directly which propagated circuit-layer error
generator each variable corresponds to. We can build a more descriptive variable-label map from the
`errorgen_to_var_map` dictionary:

In [39]:
parameter_label_map = {
    idx: f"({lbl}_{layer_idx})"
    for (lbl, layer_idx), idx in errorgen_to_var_map.items()
}
list(parameter_label_map.items())[:5]

[(0, '(H(+X___)_1)'),
 (1, '(H(+Y___)_1)'),
 (2, '(H(+Z___)_1)'),
 (3, '(S(+X___)_1)'),
 (4, '(S(+Y___)_1)')]

Now we can ask the polynomial to render itself using these custom labels.

In [40]:
print(custom_label_poly.to_string(var_labels=parameter_label_map))

2.000(H(+__Y_)_1)^2 + 4.000(H(+__Y_)_1)(H(+__Y_)_2) + 4.000(H(+__Y_)_1)(H(+__Y_)_3) + 4.000(H(+__Y_)_1)(H(+__XX)_4) + 4.000(H(+__Y_)_1)(H(+__YZ)_4) + 2.000(H(+__Z_)_1)^2 + 4.000(H(+__Z_)_1)(H(+__X_)_2) + -4.000(H(+__Z_)_1)(H(+__Z_)_3) + -4.000(H(+__Z_)_1)(H(+___Y)_4) + -4.000(H(+__Z_)_1)(H(+__Z_)_4) + 2.000(S(+__Y_)_1) + -2.000(S(+__Y_)_1)^2 + -4.000(S(+__Y_)_1)(S(+__Z_)_1) + -4.000(S(+__Y_)_1)(S(+__X_)_2) + -4.000(S(+__Y_)_1)(S(+__Y_)_2) + -4.000(S(+__Y_)_1)(S(+__Y_)_3) + -4.000(S(+__Y_)_1)(S(+__Z_)_3) + -4.000(S(+__Y_)_1)(S(+___X)_4) + -4.000(S(+__Y_)_1)(S(+___Y)_4) + -4.000(S(+__Y_)_1)(S(+__XX)_4) + -4.000(S(+__Y_)_1)(S(+__XY)_4) + -4.000(S(+__Y_)_1)(S(+__Y_)_4) + -4.000(S(+__Y_)_1)(S(+__YZ)_4) + -4.000(S(+__Y_)_1)(S(+__Z_)_4) + -4.000(S(+__Y_)_1)(S(+__ZZ)_4) + 2.000(S(+__Z_)_1) + -2.000(S(+__Z_)_1)^2 + -4.000(S(+__Z_)_1)(S(+__X_)_2) + -4.000(S(+__Z_)_1)(S(+__Y_)_2) + -4.000(S(+__Z_)_1)(S(+__Y_)_3) + -4.000(S(+__Z_)_1)(S(+__Z_)_3) + -4.000(S(+__Z_)_1)(S(+___X)_4) + -4.000(S(+__Z_)_1

Please note that `var_labels` can be supplied in several forms supported by the `Polynomial` class,
including a dictionary (as above), a sequence, or a callable. For most `errgenpolytools`
applications, a dictionary keyed by variable index is often the most convenient option.


### Bulk expectation corrections
As with probabilities, a bulk interface is available when expectation corrections for many Pauli
observables are desired.

In [41]:
paulis_4Q = np.fromiter(stim.PauliString.iter_all(num_qubits=4, min_weight=1), dtype=object)
rng = np.random.default_rng(1234)
random_paulis = rng.choice(paulis_4Q, 8, replace=False).tolist()

bulk_pauli_corr_polys = epoly.bulk_stabilizer_pauli_expectation_correction_symbolic_polynomial(
    first_order_magnus_polys, errorgen_to_var_map, tableau, random_paulis, order=1
)

In [42]:
for p, poly in zip(random_paulis[:5], bulk_pauli_corr_polys[:5]):
    print(p, "->", poly)

+Y_Z_ -> 0
+XXX_ -> 0
+_YY_ -> 0
+ZXYZ -> 2.000x14 + 2.000x36 + -2.000x62 + -2.000x85 + -2.000x95
+ZYYZ -> 0


## Aggregating Variables by Gate or Shared Model Parameters
So far we have treated each propagated input error generator occurrence as an independent polynomial
variable. This is not always the most useful choice.

In many cases we instead want polynomial variables that reflect:
- the parameters associated with a given gate occurrence, or
- shared model parameters reused across multiple gate instances.

The helper function `error_generator_to_polynomial_variable_maps_by_gate` supports this.

### Aggregating by gate
Starting from the raw variable map, we can aggregate variables by gate structure.

In [43]:
errorgen_to_var_map_gate_noagg, var_to_errorgen_map_gate_noagg = epoly.error_generator_to_polynomial_variable_maps_by_gate(
    error_model,
    errorgen_to_var_map,
    c,
    include_spam=True,
    aggregate_shared_parameter_gates=False,
    return_reverse=True
)

In [44]:
print("Number of unaggregated variables:", len(var_to_errorgen_map))
print("Number of gate-aggregated variables:", len(var_to_errorgen_map_gate_noagg))

Number of unaggregated variables: 114
Number of gate-aggregated variables: 72


### Aggregating across shared-parameter gates
Because our example noisy model shares parameters across all instances of each gate type, we can
further aggregate variables across gates that share the same underlying model parameters.

In [45]:
errorgen_to_var_map_gate_agg, var_to_errorgen_map_gate_agg = epoly.error_generator_to_polynomial_variable_maps_by_gate(
    error_model,
    errorgen_to_var_map,
    c,
    include_spam=True,
    aggregate_shared_parameter_gates=True,
    return_reverse=True
)

In [46]:
print("Number of gate-aggregated variables without shared-parameter merging:", len(var_to_errorgen_map_gate_noagg))
print("Number of gate-aggregated variables with shared-parameter merging:", len(var_to_errorgen_map_gate_agg))

Number of gate-aggregated variables without shared-parameter merging: 72
Number of gate-aggregated variables with shared-parameter merging: 42


This can significantly reduce the number of variables appearing in the resulting symbolic polynomials.

### Constructing the matching parameter vector
When using an aggregated map, it is important to evaluate the polynomials on the corresponding aggregated parameter vector.

In [47]:
poly_paramvec_gate_agg = epoly.construct_polynomial_parameter_vector_from_propagator(
    errorgen_propagator, var_to_errorgen_map_gate_agg, c
)
poly_paramvec_gate_agg

array([-0.01358528, -0.0115617 ,  0.00323786,  0.00088949,  0.00074695,
        0.00047002, -0.00909665,  0.00265269, -0.01022362,  0.00031697,
        0.00084417,  0.00030722, -0.01603837,  0.000641  ,  0.00740891,
        0.00152619,  0.00863744,  0.02913099, -0.01478823,  0.00945473,
       -0.01666135,  0.00343745, -0.00512444,  0.01323759, -0.0086028 ,
        0.00519493, -0.01265144,  0.00241433,  0.00368361,  0.00338272,
        0.00266173,  0.00373717,  0.0031591 ,  0.00415136,  0.00254938,
        0.00301312,  0.0022756 ,  0.00325679,  0.00391747,  0.00214907,
        0.0027172 ,  0.00272332])

We can now construct a first-order shared-parameter-aware symbolic Magnus approximation and evaluate it on the aggregated parameter vector.

In [48]:
first_order_magnus_polys_gate_agg = epoly.magnus_symbolic_polynomial(
    errorgen_transform_maps,
    errorgen_to_var_map_gate_agg,
    magnus_order=1
)

In [49]:
for lbl, poly in list(first_order_magnus_polys_gate_agg.items())[:5]:
    print(lbl)
    print("  polynomial:", poly)
    print("  evaluated value:", poly.evaluate(poly_paramvec_gate_agg))

H(+Z___)
  polynomial: -1.000x0 + 1.000x2 + -2.000x6
  evaluated value: (0.03501645162848567+0j)
H(+Y___)
  polynomial: 1.000x7 + -1.000x8
  evaluated value: (0.012876308836241187+0j)
H(+X___)
  polynomial: 1.000x0 + -1.000x2 + 1.000x7 + 1.000x8
  evaluated value: (-0.02439407675380065+0j)
S(+Z___)
  polynomial: 1.000x3 + 1.000x5 + 2.000x9
  evaluated value: (0.0019934452674093546+0j)
S(+Y___)
  polynomial: 2.000x4 + 1.000x10 + 1.000x11
  evaluated value: (0.0026453017369992035+0j)


## End-to-End Example Workflow
A typical `errgenpolytools` workflow looks like this:

1. Build a noisy error-generator-parameterized model and a Clifford circuit.
2. Construct propagation transform maps using `ErrorGeneratorPropagator`.
3. Build a polynomial variable map.
4. Construct symbolic Magnus polynomials.
5. Build symbolic observable correction polynomials.
6. Construct the corresponding parameter vector.
7. Evaluate the resulting polynomials.

Below is a concise example following exactly these steps.

In [50]:
errorgen_transform_maps_demo = errorgen_propagator.errorgen_transform_maps(c)
errorgen_to_var_map_demo, var_to_errorgen_map_demo = epoly.error_generator_to_polynomial_variable_maps(
    errorgen_propagator.errorgen_transform_map(c), return_reverse=True
)
magnus_polys_demo = epoly.magnus_symbolic_polynomial(
    errorgen_transform_maps_demo, errorgen_to_var_map_demo, magnus_order=1
)
prob_poly_demo = epoly.stabilizer_probability_correction_symbolic_polynomial(
    magnus_polys_demo, errorgen_to_var_map_demo, tableau, '0000', order=2
)
paramvec_demo = epoly.construct_polynomial_parameter_vector_from_propagator(
    errorgen_propagator, var_to_errorgen_map_demo, c
)
print("Evaluated probability correction:", prob_poly_demo.evaluate(paramvec_demo))

Evaluated probability correction: (0.0006329645074924+0j)


## Limitations and Current Scope
The present polynomial tools inherit several of the practical limitations of the error generator
propagation framework:

- They require Clifford-only circuits for the propagation step.
- The most useful regime is one in which the underlying error generator rates are relatively small, so that low-order BCH and Taylor approximations remain accurate.
- Symbolic expressions can grow significantly in size as one increases approximation order or circuit size.
- The symbolic Magnus implementation currently supports first and second order.
- Aggregation across shared model parameters currently assumes sufficiently local noise structure for the most advanced grouping modes.